### Cross-Domain Fake News Detection using SBERT

This notebook evaluates cross-domain generalization of fake news
classification models using Sentence-BERT embeddings.

The model is trained on the ISOT dataset and tested on the LIAR dataset
to analyze the impact of domain shift.

Pipeline:
Text → Regex Preprocessing → SBERT Embeddings → Logistic Regression

#### Experiment Setup

Training Dataset:
ISOT Fake News Dataset (news articles)

Testing Dataset:
LIAR Dataset (short political statements)

Feature Representation:
Sentence-BERT embeddings

Classifier:
Logistic Regression

Goal:
Evaluate whether semantic embeddings improve cross-domain
generalization compared to TF-IDF models.

In [1]:
import pandas as pd
df = pd.read_csv('fake_and_real_news.csv')
print(df.shape)
df.head()

(9900, 2)


,Text,label
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake
1,U.S. conservative leader optimistic of common ...,Real
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real
3,Court Forces Ohio To Allow Millions Of Illega...,Fake
4,Democrats say Trump agrees to work on immigrat...,Real


In [2]:
df['label_num'] = df['label'].map({'Fake':0,'Real':1})

In [3]:
df.head()

,Text,label,label_num
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0
1,U.S. conservative leader optimistic of common ...,Real,1
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0
4,Democrats say Trump agrees to work on immigrat...,Real,1


#### Text Preprocessing

Text cleaning is performed using regular expressions.

The following steps are applied:

• Convert text to lowercase
• Remove URLs
• Remove special characters
• Remove extra whitespace

Example regex patterns:

http\S+  → remove URLs
[^a-zA-Z\s] → remove punctuation and numbers
\s+ → normalize whitespace

In [4]:
import re

def preprocess(text):
    text = text.lower()                         
    text = re.sub(r"http\S+", "", text)        
    text = re.sub(r"[^a-zA-Z\s]", " ", text) 
    text = re.sub(r"\s+", " ", text)          
    return text.strip()

In [5]:
df['processed_text'] = df['Text'].map(preprocess)

In [6]:
df.head()

,Text,label,label_num,processed_text
0,Top Trump Surrogate BRUTALLY Stabs Him In The...,Fake,0,top trump surrogate brutally stabs him in the ...
1,U.S. conservative leader optimistic of common ...,Real,1,u s conservative leader optimistic of common g...
2,"Trump proposes U.S. tax overhaul, stirs concer...",Real,1,trump proposes u s tax overhaul stirs concerns...
3,Court Forces Ohio To Allow Millions Of Illega...,Fake,0,court forces ohio to allow millions of illegal...
4,Democrats say Trump agrees to work on immigrat...,Real,1,democrats say trump agrees to work on immigrat...


In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['processed_text'],df['label_num'],test_size=0.2,random_state=2022,stratify=df.label_num)

In [8]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np

#### Sentence Embeddings using SBERT

Each text sample is converted into a dense semantic vector using
a pretrained Sentence-BERT model.

Model Used:
all-MiniLM-L6-v2

This model generates a 384-dimensional embedding that captures
contextual meaning of sentences rather than simple word frequency.

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

X_train_emb = model.encode(X_train.tolist())
X_test_emb = model.encode(X_test.tolist())

#### Classification Model

Classifier Used:
Logistic Regression

The classifier is trained on SBERT embeddings generated from
the ISOT dataset and evaluated on the LIAR dataset.

In [10]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_emb, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [11]:
test_df = pd.read_csv('test2.tsv',sep='\t',header=None)
print(test_df.shape)
test_df.head()

(1267, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0,11972.json,true,Building a wall on the U.S.-Mexico border will...,immigration,rick-perry,Governor,Texas,republican,30,30,42,23,18,Radio interview,"Meantime, engineering experts agree the wall w..."
1,1,11685.json,false,Wisconsin is on pace to double the number of l...,jobs,katrina-shankland,State representative,Wisconsin,democrat,2,1,0,0,0,a news conference,She cited layoff notices received by the state...
2,2,11096.json,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record",donald-trump,President-Elect,New York,republican,63,114,51,37,61,comments on ABC's This Week.,"Trump said that McCain ""has done nothing to he..."
3,3,5209.json,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti...",rob-cornilles,consultant,Oregon,republican,1,1,3,1,1,a radio show,"But spending still goes up. In addition, many ..."
4,4,9524.json,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti...",state-democratic-party-wisconsin,NaN,Wisconsin,democrat,5,7,2,2,7,a web video,Our rating A Democratic Party web video making...


In [12]:
test_df = test_df[[2,3]]
test_df.columns = ['label','text']
test_df.head()

,label,text
0,true,Building a wall on the U.S.-Mexico border will...
1,false,Wisconsin is on pace to double the number of l...
2,false,Says John McCain has done nothing to help the ...
3,half-true,Suzanne Bonamici supports a plan that will cut...
4,pants-fire,When asked by a reporter whether hes at the ce...


In [13]:
test_df = test_df[test_df['label'].isin(['false','pants-fire','true','mostly-true'])]
test_df['label_num'] = test_df['label'].apply(lambda x:1 if x in ['true','mostly-true'] else 0)

In [14]:
test_df['processed_text'] = test_df['text'].apply(preprocess)

In [15]:
test_df.head()

,label,text,label_num,processed_text
0,true,Building a wall on the U.S.-Mexico border will...,1,building a wall on the u s mexico border will ...
1,false,Wisconsin is on pace to double the number of l...,0,wisconsin is on pace to double the number of l...
2,false,Says John McCain has done nothing to help the ...,0,says john mccain has done nothing to help the ...
4,pants-fire,When asked by a reporter whether hes at the ce...,0,when asked by a reporter whether hes at the ce...
5,true,Over the past five years the federal governmen...,1,over the past five years the federal governmen...


In [16]:
x_test = test_df['processed_text']
y_test =  test_df['label_num']

In [17]:
x_test_emb = model.encode(
    x_test.tolist(),
    batch_size=64,
    show_progress_bar=True
)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [18]:
y_pred = clf.predict(x_test_emb)

In [19]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.5481012658227848
Macro F1: 0.49645682576581307
Weighted F1: 0.5185026736952714
              precision    recall  f1-score   support

           0       0.46      0.26      0.34       341
           1       0.58      0.76      0.66       449

    accuracy                           0.55       790
   macro avg       0.52      0.51      0.50       790
weighted avg       0.53      0.55      0.52       790

[[ 90 251]
 [106 343]]


#### Results
Accuracy: 0.548

Macro F1: 0.496
Weighted F1: 0.518

#### Observations

The model achieves approximately 54–55% accuracy when trained on
the ISOT dataset and tested on the LIAR dataset.

Compared to TF-IDF cross-domain models (~48–51% accuracy), SBERT
embeddings provide a slight improvement in performance.

This improvement occurs because SBERT captures semantic relationships
between sentences rather than relying only on word frequency.

However, the model still struggles with domain shift due to
significant differences between the datasets:

• ISOT contains long news articles.
• LIAR contains short political claims.

The results demonstrate that while semantic embeddings help,
cross-domain fake news detection remains a challenging task.